# S3.2 — Null-Distribution Visualization & Analysis

This notebook works directly with the raw permutation null distributions
stored in `_perm_phase1.npz` (|Pearson|, |Spearman|, η²) and
`_perm_phase2.npz` (distance correlation).

**Structure:**
1. Load & inspect the raw NPZ checkpoint data
2. Visualize null distributions for selected cases
3. Null distribution shape analysis (normality, skewness)
4. Null distribution properties across case parameters
5. Deep dive: heteroscedastic vs constant-noise Null cases
6. Analysis: Z-score calibration & alternative tests

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from scipy import stats

plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 150,
                     'figure.facecolor': 'white', 'font.size': 10})

DATA_DIR = Path('generated_scatterplot_data')
OUT_DIR  = DATA_DIR / 'full' / 'S3'
VIZ_DIR  = OUT_DIR / 'viz_analysis'
VIZ_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output → {VIZ_DIR}')

## 1. Load Raw NPZ Data

In [ ]:
# Load checkpoints
p1 = np.load(OUT_DIR / '_perm_phase1.npz')
p2 = np.load(OUT_DIR / '_perm_phase2.npz')

pearson_obs  = p1['pearson_obs']    # (114176,)
spearman_obs = p1['spearman_obs']
eta2_obs     = p1['eta2_obs']
pearson_null  = p1['pearson_null']   # (114176, 500)
spearman_null = p1['spearman_null']
eta2_null     = p1['eta2_null']

dcor_obs  = p2['dcor_obs']
dcor_null = p2['dcor_null']

cases = pd.read_csv(DATA_DIR / 'cases.csv', low_memory=False)
n_cases, n_perm = pearson_null.shape

print(f'Cases: {n_cases:,}  |  Permutations: {n_perm}')
print(f'Phase 1 keys: {list(p1.files)}')
print(f'Phase 2 keys: {list(p2.files)}')
print()
for name, obs, null in [('|Pearson|', pearson_obs, pearson_null),
                         ('|Spearman|', spearman_obs, spearman_null),
                         ('η²', eta2_obs, eta2_null),
                         ('dcor', dcor_obs, dcor_null)]:
    print(f'{name:12s}  obs: [{obs.min():.4f}, {obs.max():.4f}]  '
          f'null: [{null.min():.4f}, {null.max():.4f}]  '
          f'null_mean={null.mean():.4f}')

## 2. Visualize Null Distributions for Selected Cases

Pick representative cases and show the 500-permutation null histogram
with the observed value as a vertical line.

In [ ]:
# Select 6 representative cases
null_mask = cases['family_id'] == 'Null'
null_const = cases[null_mask & (cases['spread_pattern'] == 'constant')].index
null_hetero = cases[null_mask & (cases['spread_pattern'] != 'constant')].index

sig_strong = cases[(cases['family_id'] == 'F01') & (cases['snr'] == np.inf) &
                   (cases['spread_pattern'] == 'constant')].index
sig_weak   = cases[(cases['family_id'] == 'F20') & (cases['snr'] == 0.1) &
                   (cases['spread_pattern'] == 'constant')].index
sig_nonlin = cases[(cases['family_id'] == 'F09') & (cases['snr'] == 1.0) &
                   (cases['spread_pattern'] == 'constant')].index
sig_mid    = cases[(cases['family_id'] == 'F05') & (cases['snr'] == 3.0) &
                   (cases['spread_pattern'] == 'constant')].index

selected = {}
if len(null_const) > 0:  selected['Null (constant)']    = null_const[0]
if len(null_hetero) > 0: selected['Null (heterosced.)'] = null_hetero[0]
if len(sig_strong) > 0:  selected['F01 linear SNR=∞']   = sig_strong[0]
if len(sig_weak) > 0:    selected['F20 weak SNR=0.1']   = sig_weak[0]
if len(sig_nonlin) > 0:  selected['F09 nonlin SNR=1']   = sig_nonlin[0]
if len(sig_mid) > 0:     selected['F05 mid SNR=3']      = sig_mid[0]

print(f'Selected {len(selected)} cases:')
for label, idx in selected.items():
    r = cases.iloc[idx]
    print(f'  {label:25s}  idx={idx:>6d}  family={r.family_id}  snr={r.snr}  spread={r.spread_pattern}')

In [ ]:
# Plot: 6 cases × 4 metrics = 6×4 grid of null-distribution histograms
metric_info = [
    ('|Pearson|',  pearson_obs,  pearson_null),
    ('|Spearman|', spearman_obs, spearman_null),
    ('dcor',       dcor_obs,     dcor_null),
    ('η²',        eta2_obs,     eta2_null),
]

fig, axes = plt.subplots(len(selected), 4, figsize=(18, 3*len(selected)))
fig.suptitle('Null Distributions (500 permutations) with Observed Value', fontsize=14, y=1.01)

for row, (label, idx) in enumerate(selected.items()):
    for col, (mname, obs_arr, null_arr) in enumerate(metric_info):
        ax = axes[row, col]
        null_vals = null_arr[idx].astype(np.float64)
        obs_val   = float(obs_arr[idx])

        ax.hist(null_vals, bins=40, color='steelblue', alpha=0.7, edgecolor='white', density=True)
        ax.axvline(obs_val, color='red', lw=2, label=f'obs={obs_val:.3f}')

        med = np.median(null_vals)
        ax.axvline(med, color='orange', lw=1, ls='--', label=f'med={med:.3f}')

        p_perm = (np.sum(null_vals >= obs_val) + 1) / (n_perm + 1)
        ax.set_title(f'{mname}  p={p_perm:.3f}', fontsize=9)
        ax.legend(fontsize=7, loc='upper right')

        if col == 0:
            ax.set_ylabel(label, fontsize=9)

plt.tight_layout()
fig.savefig(VIZ_DIR / '1_null_distributions_selected.png', bbox_inches='tight')
plt.show()
print(f'Saved → {VIZ_DIR}/1_null_distributions_selected.png')

## 3. Null Distribution Shape Analysis

Are the null distributions Gaussian? Check skewness, kurtosis, and
Shapiro-Wilk normality across a sample of cases.

In [ ]:
# Sample 2000 cases for shape analysis
rng = np.random.default_rng(42)
sample_idx = rng.choice(n_cases, size=min(2000, n_cases), replace=False)

shape_records = []
for idx in sample_idx:
    for mname, null_arr in [('pearson', pearson_null), ('spearman', spearman_null),
                             ('dcor', dcor_null), ('eta2', eta2_null)]:
        vals = null_arr[idx].astype(np.float64)
        skew = float(stats.skew(vals))
        kurt = float(stats.kurtosis(vals))
        _, sw_p = stats.shapiro(vals[:50])  # subsample for speed
        shape_records.append({
            'idx': idx, 'metric': mname,
            'mean': vals.mean(), 'std': vals.std(),
            'skewness': skew, 'kurtosis': kurt,
            'shapiro_p': sw_p,
        })

shape_df = pd.DataFrame(shape_records)
print('Null distribution shape statistics (sampled 2000 cases × 4 metrics):')
print(shape_df.groupby('metric')[['skewness','kurtosis','shapiro_p']].describe()
      .round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for mname, color in [('pearson','#1f77b4'), ('spearman','#ff7f0e'),
                      ('dcor','#2ca02c'), ('eta2','#d62728')]:
    sub = shape_df[shape_df['metric'] == mname]
    axes[0].hist(sub['skewness'], bins=50, alpha=0.5, label=mname, density=True)
    axes[1].hist(sub['kurtosis'], bins=50, alpha=0.5, label=mname, density=True)
    axes[2].hist(np.log10(sub['shapiro_p'].clip(1e-30)), bins=50, alpha=0.5, label=mname, density=True)

axes[0].set_title('Skewness of null distributions')
axes[0].axvline(0, color='k', ls='--', lw=0.8)
axes[0].set_xlabel('Skewness')
axes[1].set_title('Excess kurtosis of null distributions')
axes[1].axvline(0, color='k', ls='--', lw=0.8)
axes[1].set_xlabel('Kurtosis')
axes[2].set_title('Shapiro-Wilk p-value (log₁₀)')
axes[2].axvline(np.log10(0.05), color='k', ls='--', lw=0.8, label='p=0.05')
axes[2].set_xlabel('log₁₀(p)')

for ax in axes:
    ax.legend(fontsize=8)

plt.suptitle('Shape of Null Distributions (n=500 permutations each)', fontsize=13)
plt.tight_layout()
fig.savefig(VIZ_DIR / '2_null_shape_analysis.png', bbox_inches='tight')
plt.show()
print(f'Saved → {VIZ_DIR}/2_null_shape_analysis.png')

## 4. Null Distribution Properties Across Case Parameters

How do the null distribution median and IQR vary with SNR, family, spread_pattern?

In [ ]:
# Compute null median and IQR for all cases, all metrics
null_props = pd.DataFrame({'case_idx': np.arange(n_cases)})

for mname, null_arr in [('pearson', pearson_null), ('spearman', spearman_null),
                         ('dcor', dcor_null), ('eta2', eta2_null)]:
    null_f64 = null_arr.astype(np.float64)
    null_props[f'{mname}_null_med'] = np.median(null_f64, axis=1)
    q75 = np.percentile(null_f64, 75, axis=1)
    q25 = np.percentile(null_f64, 25, axis=1)
    null_props[f'{mname}_null_iqr'] = q75 - q25
    null_props[f'{mname}_null_max'] = null_f64.max(axis=1)
    null_props[f'{mname}_null_std'] = null_f64.std(axis=1)

null_props['family_id'] = cases['family_id'].values
null_props['snr'] = cases['snr'].values
null_props['spread_pattern'] = cases['spread_pattern'].values
null_props['x_distribution'] = cases['x_distribution'].values

print(null_props.describe().round(4).to_string())

In [ ]:
# Null median and IQR by x_distribution (should affect null baseline)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Null Distribution Properties by x_distribution', fontsize=13)

for col, mname in enumerate(['pearson', 'spearman', 'dcor', 'eta2']):
    for xd in sorted(null_props['x_distribution'].unique()):
        sub = null_props[null_props['x_distribution'] == xd]
        axes[0, col].hist(sub[f'{mname}_null_med'], bins=50, alpha=0.4, label=xd, density=True)
        axes[1, col].hist(sub[f'{mname}_null_iqr'], bins=50, alpha=0.4, label=xd, density=True)

    axes[0, col].set_title(f'{mname} — null median')
    axes[1, col].set_title(f'{mname} — null IQR')
    axes[0, col].legend(fontsize=6)

plt.tight_layout()
fig.savefig(VIZ_DIR / '3_null_props_by_xdist.png', bbox_inches='tight')
plt.show()
print(f'Saved → {VIZ_DIR}/3_null_props_by_xdist.png')

In [ ]:
# Null properties by spread_pattern — critical for FP analysis
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Null Distribution Properties by spread_pattern', fontsize=13)
colors_sp = {'constant': '#2ca02c', 'increasing': '#d62728',
             'decreasing': '#1f77b4', 'middle_high': '#ff7f0e'}

for col, mname in enumerate(['pearson', 'spearman', 'dcor', 'eta2']):
    for sp, c in colors_sp.items():
        sub = null_props[null_props['spread_pattern'] == sp]
        axes[0, col].hist(sub[f'{mname}_null_med'], bins=50, alpha=0.4, label=sp, color=c, density=True)
        axes[1, col].hist(sub[f'{mname}_null_iqr'], bins=50, alpha=0.4, label=sp, color=c, density=True)

    axes[0, col].set_title(f'{mname} — null median')
    axes[1, col].set_title(f'{mname} — null IQR')
    axes[0, col].legend(fontsize=7)

plt.tight_layout()
fig.savefig(VIZ_DIR / '4_null_props_by_spread.png', bbox_inches='tight')
plt.show()
print(f'Saved → {VIZ_DIR}/4_null_props_by_spread.png')

## 5. Deep Dive: Heteroscedastic vs Constant Null Cases

The key finding: dcor detects heteroscedastic noise as real dependence.
Compare the actual null distributions side by side.

In [ ]:
# Find all Null cases and compare their null distributions
null_cases = cases[cases['family_id'] == 'Null'].copy()
null_cases['case_idx'] = null_cases.index.values
print(f'Total Null cases: {len(null_cases)}')
print(null_cases['spread_pattern'].value_counts().to_string())
print()

# For each Null case: obs, null_med, null_max, per-metric p-value
null_detail = []
for _, row in null_cases.iterrows():
    idx = row['case_idx']
    rec = {'idx': idx, 'spread_pattern': row['spread_pattern'],
           'x_distribution': row['x_distribution']}
    for mname, obs_arr, null_arr in [('pearson', pearson_obs, pearson_null),
                                      ('spearman', spearman_obs, spearman_null),
                                      ('dcor', dcor_obs, dcor_null),
                                      ('eta2', eta2_obs, eta2_null)]:
        o = float(obs_arr[idx])
        n = null_arr[idx].astype(np.float64)
        rec[f'{mname}_obs'] = o
        rec[f'{mname}_null_med'] = np.median(n)
        rec[f'{mname}_null_q95'] = np.percentile(n, 95)
        rec[f'{mname}_null_max'] = n.max()
        rec[f'{mname}_perm_p'] = (np.sum(n >= o) + 1) / (n_perm + 1)
    null_detail.append(rec)

null_det_df = pd.DataFrame(null_detail)
print('Per-metric permutation p-value summary for Null cases:')
for sp in ['constant', 'increasing', 'decreasing', 'middle_high']:
    sub = null_det_df[null_det_df['spread_pattern'] == sp]
    print(f'\n  spread={sp} (n={len(sub)}):')
    for m in ['pearson', 'spearman', 'dcor', 'eta2']:
        fp = (sub[f'{m}_perm_p'] <= 0.05).mean()
        print(f'    {m:10s}  FP rate={fp:.1%}   median_p={sub[f"{m}_perm_p"].median():.3f}')

In [ ]:
# Deep dive: dcor null distributions for constant vs heteroscedastic Null cases
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('dcor: Null Distribution Comparison — Constant vs Heteroscedastic', fontsize=13)

# Top row: 4 constant-noise Null cases
const_null_idx = null_cases[null_cases['spread_pattern'] == 'constant']['case_idx'].values
for i, idx in enumerate(const_null_idx[:4]):
    ax = axes[0, i]
    null_vals = dcor_null[idx].astype(np.float64)
    obs_val = float(dcor_obs[idx])
    ax.hist(null_vals, bins=35, color='#2ca02c', alpha=0.7, edgecolor='white', density=True)
    ax.axvline(obs_val, color='red', lw=2, label=f'obs={obs_val:.3f}')
    p = (np.sum(null_vals >= obs_val) + 1) / (n_perm + 1)
    xd = cases.iloc[idx]['x_distribution']
    ax.set_title(f'Null const, {xd}\np={p:.3f}', fontsize=9)
    ax.legend(fontsize=7)
    if i == 0: ax.set_ylabel('constant noise', fontsize=10)

# Bottom row: 4 heteroscedastic Null cases (one per spread type)
for i, sp in enumerate(['increasing', 'decreasing', 'middle_high']):
    hetero_idx = null_cases[null_cases['spread_pattern'] == sp]['case_idx'].values
    if len(hetero_idx) == 0: continue
    idx = hetero_idx[0]
    ax = axes[1, i]
    null_vals = dcor_null[idx].astype(np.float64)
    obs_val = float(dcor_obs[idx])
    ax.hist(null_vals, bins=35, color='#d62728', alpha=0.7, edgecolor='white', density=True)
    ax.axvline(obs_val, color='red', lw=2, label=f'obs={obs_val:.3f}')
    p = (np.sum(null_vals >= obs_val) + 1) / (n_perm + 1)
    ax.set_title(f'Null {sp}\np={p:.3f}', fontsize=9)
    ax.legend(fontsize=7)
    if i == 0: ax.set_ylabel('heterosced. noise', fontsize=10)

# Last bottom panel: overlay obs vs null for all null cases
ax = axes[1, 3]
for sp, c in [('constant','#2ca02c'), ('increasing','#d62728'),
               ('decreasing','#1f77b4'), ('middle_high','#ff7f0e')]:
    idxs = null_cases[null_cases['spread_pattern'] == sp]['case_idx'].values
    obs_vals = dcor_obs[idxs]
    null_meds = np.median(dcor_null[idxs].astype(np.float64), axis=1)
    ax.scatter(null_meds, obs_vals, s=30, alpha=0.7, label=sp, color=c)
ax.plot([0, 0.25], [0, 0.25], 'k--', lw=0.8)
ax.set_xlabel('null median')
ax.set_ylabel('observed dcor')
ax.set_title('obs vs null median (all Null cases)')
ax.legend(fontsize=7)

plt.tight_layout()
fig.savefig(VIZ_DIR / '5_dcor_null_deep_dive.png', bbox_inches='tight')
plt.show()
print(f'Saved → {VIZ_DIR}/5_dcor_null_deep_dive.png')

## 6. Observed vs Null: All Cases

Scatter obs value against null median/max for all 114K cases. Separation
= detection power.

In [ ]:
# obs vs null_median and obs vs null_max for each metric
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Observed vs Null Distribution Statistics (5K sample)', fontsize=13)

sample_idx2 = rng.choice(n_cases, size=5000, replace=False)
is_null = cases.iloc[sample_idx2]['family_id'].values == 'Null'

for col, (mname, obs_arr, null_arr) in enumerate(metric_info):
    obs_s  = obs_arr[sample_idx2]
    n_f64  = null_arr[sample_idx2].astype(np.float64)
    n_med  = np.median(n_f64, axis=1)
    n_max  = n_f64.max(axis=1)

    ax = axes[0, col]
    ax.scatter(n_med[~is_null], obs_s[~is_null], s=1, alpha=0.3, c='steelblue', label='Signal')
    ax.scatter(n_med[is_null], obs_s[is_null], s=20, alpha=0.9, c='red', marker='x', label='Null')
    ax.plot([0, 1], [0, 1], 'k--', lw=0.5)
    ax.set_xlabel('null median'); ax.set_ylabel('observed')
    ax.set_title(f'{mname} — obs vs null median')
    ax.legend(fontsize=7)

    ax = axes[1, col]
    ax.scatter(n_max[~is_null], obs_s[~is_null], s=1, alpha=0.3, c='steelblue', label='Signal')
    ax.scatter(n_max[is_null], obs_s[is_null], s=20, alpha=0.9, c='red', marker='x', label='Null')
    ax.plot([0, 1], [0, 1], 'k--', lw=0.5)
    ax.set_xlabel('null max'); ax.set_ylabel('observed')
    ax.set_title(f'{mname} — obs vs null max')
    ax.legend(fontsize=7)

plt.tight_layout()
fig.savefig(VIZ_DIR / '6_obs_vs_null_scatter.png', bbox_inches='tight')
plt.show()
print(f'Saved → {VIZ_DIR}/6_obs_vs_null_scatter.png')

## 7. Z-Score Calibration Analysis

Recompute Z-scores from raw data and check if the median/IQR
normalization produces well-calibrated scores under the null.

In [ ]:
# For all Null cases: compute Z-score of each null permutation
# under the null, Z should be ~ standard (median 0, IQR ~1.35)

null_idx_all = null_cases['case_idx'].values
const_idx = null_cases[null_cases['spread_pattern'] == 'constant']['case_idx'].values
hetero_idx = null_cases[null_cases['spread_pattern'] != 'constant']['case_idx'].values

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Z-score Calibration Under the Null (Null cases only)', fontsize=13)

for col, (mname, obs_arr, null_arr) in enumerate(metric_info):
    # Constant noise null cases
    all_z_const = []
    for idx in const_idx:
        null_vals = null_arr[idx].astype(np.float64)
        med = np.median(null_vals)
        iqr = np.percentile(null_vals, 75) - np.percentile(null_vals, 25)
        if iqr < 1e-12:
            iqr = np.std(null_vals) * 1.35
        if iqr > 1e-12:
            z = (null_vals - med) / iqr
            all_z_const.extend(z.tolist())

    all_z_hetero = []
    for idx in hetero_idx:
        null_vals = null_arr[idx].astype(np.float64)
        med = np.median(null_vals)
        iqr = np.percentile(null_vals, 75) - np.percentile(null_vals, 25)
        if iqr < 1e-12:
            iqr = np.std(null_vals) * 1.35
        if iqr > 1e-12:
            z = (null_vals - med) / iqr
            all_z_hetero.extend(z.tolist())

    ax = axes[0, col]
    ax.hist(all_z_const, bins=60, density=True, alpha=0.7, color='#2ca02c', label='constant')
    xr = np.linspace(-4, 4, 200)
    ax.plot(xr, stats.norm.pdf(xr), 'k--', lw=1, label='N(0,1)')
    ax.set_title(f'{mname} — constant noise')
    ax.set_xlim(-5, 5)
    ax.legend(fontsize=7)
    if col == 0: ax.set_ylabel('Density')

    ax = axes[1, col]
    ax.hist(all_z_hetero, bins=60, density=True, alpha=0.7, color='#d62728', label='heterosced.')
    ax.plot(xr, stats.norm.pdf(xr), 'k--', lw=1, label='N(0,1)')
    ax.set_title(f'{mname} — heteroscedastic')
    ax.set_xlim(-5, 5)
    ax.legend(fontsize=7)
    if col == 0: ax.set_ylabel('Density')

plt.tight_layout()
fig.savefig(VIZ_DIR / '7_zscore_calibration.png', bbox_inches='tight')
plt.show()
print(f'Saved → {VIZ_DIR}/7_zscore_calibration.png')

## 8. Effect Size Analysis

Beyond p-values: how far is the observed value from the null?
Compute obs/null_median ratio and obs−null_q95 gap.

In [ ]:
# Effect size: obs / null_median ratio across SNR
signal_mask = cases['family_id'] != 'Null'
signal_idx = np.where(signal_mask)[0]
snr_vals = pd.to_numeric(cases.loc[signal_mask, 'snr'], errors='coerce').values

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
fig.suptitle('Effect Size: obs / null_median by SNR (Signal cases)', fontsize=13)

snr_unique = np.sort(np.unique(snr_vals[~np.isnan(snr_vals)]))

for col, (mname, obs_arr, null_arr) in enumerate(metric_info):
    ratios_by_snr = []
    for snr in snr_unique:
        mask = snr_vals == snr
        idxs = signal_idx[mask]
        if len(idxs) == 0: continue
        obs = obs_arr[idxs]
        null_med = np.median(null_arr[idxs].astype(np.float64), axis=1)
        ratio = obs / np.clip(null_med, 1e-10, None)
        ratios_by_snr.append((snr, np.median(ratio), np.percentile(ratio, 25), np.percentile(ratio, 75)))

    rs = np.array(ratios_by_snr)
    ax = axes[col]
    ax.plot(range(len(rs)), rs[:, 1], 'o-', color='steelblue', lw=2)
    ax.fill_between(range(len(rs)), rs[:, 2], rs[:, 3], alpha=0.2, color='steelblue')
    ax.set_xticks(range(len(rs)))
    ax.set_xticklabels([f'{s:.1f}' if s < 100 else ('inf' if np.isinf(s) else f'{s:.0f}') for s in rs[:, 0]],
                       rotation=45, fontsize=8)
    ax.set_xlabel('SNR')
    ax.set_ylabel('obs / null_median')
    ax.set_title(mname)
    ax.axhline(1, color='k', ls='--', lw=0.5)
    ax.set_yscale('log')

plt.tight_layout()
fig.savefig(VIZ_DIR / '8_effect_size_by_snr.png', bbox_inches='tight')
plt.show()
print(f'Saved → {VIZ_DIR}/8_effect_size_by_snr.png')

## 9. Obs vs Null Separation by Metric

For each metric: what fraction of observed values exceed
the null distribution's 95th, 99th, and max percentile?

In [ ]:
# Compute exceedance fractions
results_exc = []
for mname, obs_arr, null_arr in [('|Pearson|', pearson_obs, pearson_null),
                                  ('|Spearman|', spearman_obs, spearman_null),
                                  ('dcor', dcor_obs, dcor_null),
                                  ('η²', eta2_obs, eta2_null)]:
    null_f64 = null_arr.astype(np.float64)
    q95  = np.percentile(null_f64, 95, axis=1)
    q99  = np.percentile(null_f64, 99, axis=1)
    nmax = null_f64.max(axis=1)

    for label, subset_mask in [('All', np.ones(n_cases, bool)),
                                ('Signal', signal_mask.values),
                                ('Null', null_mask.values)]:
        idx = np.where(subset_mask)[0]
        obs = obs_arr[idx]
        results_exc.append({
            'metric': mname, 'subset': label, 'n': len(idx),
            'exc_q95': (obs > q95[idx]).mean(),
            'exc_q99': (obs > q99[idx]).mean(),
            'exc_max': (obs > nmax[idx]).mean(),
        })

exc_df = pd.DataFrame(results_exc)
print('Fraction of observed values exceeding null percentiles:')
print(exc_df.to_string(index=False, float_format='%.3f'))

In [ ]:
# Visualize: stacked bar of exceedance
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Fraction of obs Exceeding Null Percentiles', fontsize=13)

for i, subset in enumerate(['Signal', 'Null', 'All']):
    sub = exc_df[exc_df['subset'] == subset]
    x = range(len(sub))
    ax = axes[i]
    ax.bar(x, sub['exc_q95'], color='#2ca02c', alpha=0.7, label='> q95')
    ax.bar(x, sub['exc_q99'], color='#ff7f0e', alpha=0.7, label='> q99')
    ax.bar(x, sub['exc_max'], color='#d62728', alpha=0.7, label='> max')
    ax.set_xticks(x)
    ax.set_xticklabels(sub['metric'], rotation=30)
    ax.set_title(f'{subset} cases')
    ax.set_ylabel('Fraction')
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1.05)

plt.tight_layout()
fig.savefig(VIZ_DIR / '9_exceedance_fractions.png', bbox_inches='tight')
plt.show()
print(f'Saved → {VIZ_DIR}/9_exceedance_fractions.png')

## 10. Cross-Metric Null Correlation

Are the 500 null values for different metrics correlated within the same case?
High correlation → redundancy in the joint test.

In [ ]:
# For a sample of cases: correlation between null distributions of different metrics
sample_corr_idx = rng.choice(n_cases, size=500, replace=False)

corr_pairs = [('pearson-spearman', pearson_null, spearman_null),
              ('pearson-dcor', pearson_null, dcor_null),
              ('pearson-eta2', pearson_null, eta2_null),
              ('spearman-dcor', spearman_null, dcor_null),
              ('spearman-eta2', spearman_null, eta2_null),
              ('dcor-eta2', dcor_null, eta2_null)]

null_corr_records = []
for idx in sample_corr_idx:
    for pair_name, arr_a, arr_b in corr_pairs:
        a = arr_a[idx].astype(np.float64)
        b = arr_b[idx].astype(np.float64)
        r, _ = stats.pearsonr(a, b)
        null_corr_records.append({'idx': idx, 'pair': pair_name, 'r': r})

null_corr_df = pd.DataFrame(null_corr_records)
print('Cross-metric null correlation (within each case, across 500 permutations):')
print(null_corr_df.groupby('pair')['r'].describe().round(3).to_string())
print()

fig, ax = plt.subplots(figsize=(10, 5))
for pair_name in [p[0] for p in corr_pairs]:
    sub = null_corr_df[null_corr_df['pair'] == pair_name]
    ax.hist(sub['r'], bins=40, alpha=0.5, label=pair_name, density=True)
ax.set_xlabel('Pearson r between null distributions')
ax.set_ylabel('Density')
ax.set_title('Cross-Metric Null Correlation (500 sampled cases)', fontsize=12)
ax.legend(fontsize=8)
ax.axvline(0, color='k', ls='--', lw=0.5)

plt.tight_layout()
fig.savefig(VIZ_DIR / '10_null_cross_correlation.png', bbox_inches='tight')
plt.show()
print(f'Saved → {VIZ_DIR}/10_null_cross_correlation.png')

## 11. Summary Table

Aggregate all null distribution properties into a reference table.

In [ ]:
# Final summary table
summary_rows = []
for mname, obs_arr, null_arr in [('|Pearson|', pearson_obs, pearson_null),
                                  ('|Spearman|', spearman_obs, spearman_null),
                                  ('dcor', dcor_obs, dcor_null),
                                  ('η²', eta2_obs, eta2_null)]:
    null_f64 = null_arr.astype(np.float64)
    meds = np.median(null_f64, axis=1)
    iqrs = np.percentile(null_f64, 75, axis=1) - np.percentile(null_f64, 25, axis=1)

    summary_rows.append({
        'metric': mname,
        'obs_mean': obs_arr.mean(),
        'obs_std': obs_arr.std(),
        'null_med_mean': meds.mean(),
        'null_med_std': meds.std(),
        'null_iqr_mean': iqrs.mean(),
        'null_iqr_std': iqrs.std(),
        'null_overall_mean': null_f64.mean(),
        'null_overall_std': null_f64.std(),
    })

summary_df = pd.DataFrame(summary_rows)
print('=== Null Distribution Summary ===')
print(summary_df.to_string(index=False, float_format='%.4f'))
print()
print('Key observations:')
print('• All null medians are low and stable (~0.03 for Pearson/Spearman, ~0.07 for dcor, ~0.02 for η²)')
print('• Null IQRs are tight → Z-score normalization amplifies signal well')
print('• dcor has the highest null baseline — its null is not centered at 0')